# Foley backbone: yapılandırılmış olay isteği
İlk entegrasyon AudioLDM m-full kullanır. Kullanıcı prompt'u veya üretim sırasında referans WAV verilmez; `event_id` offline hazırlanmış CLAP audio prototype'ını seçer. Repodaki ilk Footstep prototype tek gerçek örnekten gelir; bu genel Foley modeli değildir. Sayısal kontrol adapter'ı için mimari/eğitim hattı var, fakat kullanıma hazır eğitilmiş checkpoint henüz yok. Aşağıdaki üretim **kontrolsüz pretrained baseline**dır.

08'in `.venv-audioldm` kurulumu ve büyük checkpoint/source hazırlığı gerekir. [Mimari kararı](../docs/decisions/001_foley_conditioning_architecture.md).

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
assert (ROOT / "src/foley").exists(), "Notebooku sound-gen/notebooks altında açın."
MODEL_PYTHON = ROOT / ".venv-audioldm" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
assert MODEL_PYTHON.exists(), "08 notebookundaki ayrı ortam kurulumunu tamamlayın."
request = json.loads((ROOT / "configs/foley_baseline_request.json").read_text())
request

`family` olay ailesi, `event_id` kayıtlı semantik koşul, `duration_s` istenen çıktı süresidir. `seed` çeşitliliği sabitlemek için kullanılır; fiziksel kontrol değildir. İlk örnekte `controls` boştur. Üretim, model ortamında subprocess olarak çalışır.

In [ ]:
subprocess.run([str(MODEL_PYTHON), str(ROOT / "src/foley/infer.py"),
                "--request", str(ROOT / "configs/foley_baseline_request.json"),
                "--device", "auto", "--output-name", "local_foley_baseline"], check=True)
import soundfile as sf
y, sr = sf.read(ROOT / "results/audio/foley/local_foley_baseline.wav")
display(Audio(y, rate=sr, normalize=False))
json.loads((ROOT / "experiments/architecture/local_foley_baseline_run.json").read_text())

Sayısal hedef isteği aşağıdadır. RMS eğrisi ve centroid modele girecek koşullardır; materyal/kuvvet gibi yeni parametrelerin registry tanımı yanında gerçek etiketli eğitim gerekir. Eğitilmiş adapter olmadan bu isteği üretime göndermek hata verir; kontroller yok sayılmaz. Eğitilmiş checkpoint geldiğinde CLI'ye `--adapter models/foley/adapter.pt` eklenir.

In [ ]:
target = json.loads((ROOT / "configs/foley_target_request.json").read_text())
target

Mevcut entegrasyon tanısı, gerçek codec latentleri ve backbone üzerinde iki optimizer adımıdır. Bu, kontrol başarısı değerlendirmesi değildir. Veri cache, train-only prototype ve adapter eğitim adımları [çalışma notunda](../experiments/architecture/README.md).

In [ ]:
json.loads((ROOT / "experiments/architecture/audioldm_adapter_smoke.json").read_text())